# Cell types from graph topology (Danenberg, Jackson, J→D, D→J)

**Question.** Following Samoilov-Katz et al. 2021 (arXiv 2106.13672), which classified bloggers' domains of interest from network topology alone: can a cell's position in the tissue graph predict its type?
- Does topology add anything to the best model of `cell-type-graph` (Table + nbr)?

**Same as `cell-type-graph`:** labels (fine; 6 coarse), splits (same seed), 15 shared inputs, LightGBM settings, class weights, patient-bootstrap scoring.

**Topology features per cell** (no marker values), each image's graph separately:
- **On the contact graph and on the 30 µm radius graph:** log degree; number of triangles; open triads with the cell at the centre and at an end (the undirected 3-node motifs); clustering coefficient; log number of cells exactly 2 hops away.
- **On the contact graph only:** harmonic closeness.
- 13 features, harmonised per cohort like the markers.
- Caveat: Danenberg's contact graph is the published one; Jackson's was built from masks at 4 px.

**Arms (all LightGBM):**
- Table;
- Table + nbr;
- **Topology:** topology only, which mirrors the paper;
- **Table + topo:** own inputs + topology;
- **Table + nbr + topo.**

CPU only (no networks).

In [ ]:
import os, glob, json, time, sys, subprocess, warnings, gc
import numpy as np, pandas as pd
import pyarrow.parquet as pq
import scipy.sparse as sp
from scipy.spatial import cKDTree
from scipy.sparse.csgraph import shortest_path
try:
    import lightgbm as lgb
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightgbm"], check=True)
    import lightgbm as lgb
import torch
import torch.nn as nn
import torch.nn.functional as F
warnings.filterwarnings("ignore")

IN = os.environ.get("IN_ROOT", "/kaggle/input")
OUT = os.environ.get("OUT_DIR", "/kaggle/working")
SMOKE = os.environ.get("SMOKE") == "1"
SEED = 20261013                                   # same seed and split rules as train-test-grid
N_BOOT = 100 if SMOKE else 1000
MIN_CELLS = 200
COF = 5.0
N_FIT = 20_000 if SMOKE else 300_000              # labelled training cells per cohort for LightGBM
LGB = dict(n_estimators=30 if SMOKE else 250, learning_rate=0.06, num_leaves=31, min_child_samples=50,
           subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, n_jobs=os.cpu_count(), verbose=-1)
HID = 32 if SMOKE else 128
MAX_EPOCHS = 3 if SMOKE else 50
PATIENCE = 2 if SMOKE else 5
BATCH_IMAGES = 16
EVAL_IMAGES = 32
BUDGET_MIN = float(os.environ.get("BUDGET_MIN", 660))
REPS = [int(r) for r in os.environ.get("REPS", "0,1,2").split(",")]
SWEEP = False
dev = "cuda" if torch.cuda.is_available() else "cpu"

NUC = ["histone_h3", "dna1", "dna2"]
MARKERS = ["c_erb_b_2_her2", "cd20", "cd3", "cd45", "cd68", "cytokeratin_5", "cytokeratin_8_18",
           "estrogen_receptor_alpha", "ki_67", "pan_cytokeratin", "sma"]
FEATS = NUC + MARKERS                             # + log area
D_TABLE = {"c_erb_b_2_her2": "c_erb_b_2_her2_3b5"}  # name -> Danenberg cell-table column

COARSE = ["epithelial", "T cell", "B cell", "myeloid", "endothelial", "stroma"]
# Danenberg: every phenotype whose cells are mostly is_epithelial -> epithelial; the rest by name; others unlabelled
D_COARSE = {"CD8^{+} T cells": "T cell", "CD4^{+} T cells": "T cell", "T_{Reg} & T_{Ex}": "T cell",
            "B cells": "B cell",
            "Macrophages": "myeloid", "Granulocytes": "myeloid", "Macrophages & granulocytes": "myeloid",
            "Endothelial": "endothelial",
            "Fibroblasts": "stroma", "Fibroblasts FSP1^{+}": "stroma", "Myofibroblasts": "stroma",
            "Myofibroblasts PDPN^{+}": "stroma"}
# Jackson by metacluster id. 1 and 2 by marker evidence (published names swapped); 3 and 5 unlabelled.
J_COARSE = {1: "T cell", 2: "B cell", 4: "myeloid", 6: "myeloid", 7: "endothelial",
            **{m: "stroma" for m in range(8, 14)}, **{m: "epithelial" for m in range(14, 28)}}

GRAPHS = {"knn10": ("knn", 10), "r30": ("radius", 30.0)}
R_CAP = 32
T0 = time.time()

def tick(msg):
    print(f"[{(time.time() - T0) / 60:6.1f} min] {msg}", flush=True)

def g(pattern):
    return sorted(glob.glob(f"{IN}/**/{pattern}", recursive=True))

def near(name, anchor):
    hits = g(name)
    assert hits, f"missing input {name}"
    return max(hits, key=lambda c: len(os.path.commonpath([c, anchor])))

## Cell tables, inputs and labels

In [ ]:
P = {}
for p in g("cells.parquet"):
    key = "Danenberg" if "metabric_id" in pq.read_schema(p).names else "Jackson"
    P[key] = dict(cells=p, contacts=near("cell_neighbours.parquet", p))
P["Danenberg"]["split"] = near("danenberg_patient_split.csv", P["Danenberg"]["cells"])
print(json.dumps(P, indent=1))

def load(key):
    cells = pd.read_parquet(P[key]["cells"])
    if key == "Danenberg":
        cells = cells[(cells.is_tumour == 1) & (cells.is_hot_aggregate == 0)]
        pid = "metabric_id"
    else:
        cells = cells[cells.is_tumour == 1]
        pid = "patient_id"
    n_per = cells.groupby("image_name").size()
    cells = cells[cells.image_name.isin(n_per.index[n_per >= MIN_CELLS])]
    cells = cells.sort_values(["image_name", "object_number"]).reset_index(drop=True)
    n = len(cells)
    img, img_names = pd.factorize(cells.image_name, sort=True)
    starts = np.r_[0, np.flatnonzero(np.diff(img)) + 1, n]
    # contact graph -> row indices
    loc = pd.Series(np.arange(n), index=pd.MultiIndex.from_arrays([cells.image_number.values, cells.object_number.values]))
    con = pd.read_parquet(P[key]["contacts"])
    ca = loc.reindex(pd.MultiIndex.from_arrays([con.ImageNumber.values, con["from"].values])).values
    cb = loc.reindex(pd.MultiIndex.from_arrays([con.ImageNumber.values, con["to"].values])).values
    ok = ~(np.isnan(ca) | np.isnan(cb))
    ca, cb = ca[ok].astype(np.int64), cb[ok].astype(np.int64)
    ca, cb = ca[ca != cb], cb[ca != cb]
    CON = sp.coo_matrix((np.ones(len(ca)), (ca, cb)), shape=(n, n)).tocsr()
    CON = ((CON + CON.T) > 0).astype(np.float64).tocsr()
    # inputs
    cols = [D_TABLE.get(c, c) for c in FEATS] if key == "Danenberg" else FEATS
    X = np.c_[np.arcsinh(cells[cols].values.astype(np.float64) / COF),
              np.log(np.maximum(cells["area"].values, 1))].astype(np.float32)
    # labels
    phen = cells.cell_phenotype.astype(str).values
    fine = sorted(set(phen))
    y_fine = pd.Series(phen).map({p: i for i, p in enumerate(fine)}).values.astype(np.int64)
    if key == "Danenberg":
        epi = cells.groupby(cells.cell_phenotype.astype(str)).is_epithelial.mean()
        cmap = {p: ("epithelial" if epi[p] > 0.5 else D_COARSE.get(p)) for p in fine}
    else:
        mc = cells.groupby(cells.cell_phenotype.astype(str)).metacluster.first()
        cmap = {p: J_COARSE.get(int(mc[p])) for p in fine}
    coarse_of = np.array([COARSE.index(cmap[p]) if cmap[p] else -1 for p in fine], np.int64)
    return dict(key=key, n=n, starts=starts, n_img=len(img_names), img_names=np.asarray(img_names),
                patient=cells[pid].astype(str).values, xy=cells[["x", "y"]].values.astype(np.float64),
                CON=CON, X=X, fine=fine, cmap=cmap, y={"fine": y_fine, "coarse": coarse_of[y_fine]})

CO = {k: load(k) for k in ("Jackson", "Danenberg")}
CJ, CD = CO["Jackson"], CO["Danenberg"]
split0 = pd.read_csv(P["Danenberg"]["split"]).set_index("metabric_id").split
CD["split0"] = pd.Series(CD["patient"]).map(split0).fillna("none").values

rows = []
for key, C in CO.items():
    print(f"{key}: {C['n']:,} cells, {C['n_img']} images, {len(set(C['patient']))} patients, {len(C['fine'])} fine labels; "
          f"coarse {pd.Series(C['y']['coarse']).map(dict(enumerate(COARSE))).fillna('unlabelled').value_counts().to_dict()}")
    for i, p in enumerate(C["fine"]):
        m = C["y"]["fine"] == i
        r = dict(cohort=key, fine=p, coarse=C["cmap"][p] or "unlabelled", n=int(m.sum()))
        r.update({f: float(np.median(C["X"][m, j])) for j, f in enumerate(FEATS)})
        rows.append(r)
pd.DataFrame(rows).to_csv(f"{OUT}/labels_check.csv", index=False)

def harm(X):
    med = np.nanmedian(X, 0)
    q05, q95 = np.nanpercentile(X, [5, 95], 0)
    sc = np.where(q95 - q05 > 1e-3, q95 - q05, np.nanstd(X, 0) + 1e-6)
    return ((X - med) / sc).astype(np.float32)

for C in CO.values():
    C["XS"] = {"cohort": np.nan_to_num(harm(C["X"])),
               "img": np.nan_to_num(np.vstack([harm(C["X"][a:b]) for a, b in zip(C["starts"][:-1], C["starts"][1:])]))}
tick("tables loaded")

## Graphs and neighbour features

In [ ]:
def adjacency(C, kind, par):
    rows, cols = [], []
    for a, b in zip(C["starts"][:-1], C["starts"][1:]):
        xy = C["xy"][a:b]
        k = min(par if kind == "knn" else R_CAP, b - a - 1)
        if kind == "knn":
            d, nb = cKDTree(xy).query(xy, k=k + 1)
        else:
            d, nb = cKDTree(xy).query(xy, k=k + 1, distance_upper_bound=par)
        d, nb = d[:, 1:], nb[:, 1:]
        ok = np.isfinite(d)
        rows.append(np.repeat(np.arange(a, b), k)[ok.ravel()])
        cols.append(nb[ok] + a)
    rows, cols = np.concatenate(rows), np.concatenate(cols)
    A = sp.coo_matrix((np.ones(len(rows), np.float32), (rows, cols)), shape=(C["n"], C["n"])).tocsr()
    A = ((A + A.T) > 0).astype(np.float32).tocsr()
    A.sort_indices()
    return A

def edge_store(C, A, with_attr):
    """Edges grouped by receiving cell (dst); dst lies in the same image as src, so edges are grouped by image."""
    dst = np.repeat(np.arange(C["n"], dtype=np.int32), np.diff(A.indptr))
    src = A.indices.astype(np.int32)
    E = dict(src=src, dst=dst, estarts=A.indptr[C["starts"]].astype(np.int64))
    st = dict(mean_degree=float(np.diff(A.indptr).mean()))
    if with_attr:
        dist = np.linalg.norm(C["xy"][src] - C["xy"][dst], axis=1)
        Ai = sp.csr_matrix((np.arange(1, A.nnz + 1, dtype=np.float64), A.indices, A.indptr), shape=A.shape)
        hit = Ai.multiply(C["CON"]).tocsr()
        con = np.zeros(A.nnz, np.float32)
        con[(hit.data[hit.data > 0] - 1).astype(np.int64)] = 1
        E["ea"] = np.c_[dist / 30.0, con].astype(np.float32)
        st.update(median_distance=float(np.median(dist)), contact_fraction=float(con.mean()))
    return E, st

def nbr_features(C, A):
    X = C["XS"]["cohort"]
    deg = np.diff(A.indptr).astype(np.float32)[:, None]            # every kNN row has >= k neighbours
    mean = (A @ X) / deg
    std = np.sqrt(np.maximum((A @ (X * X)) / deg - mean * mean, 0))
    mx = np.maximum.reduceat(X[A.indices], A.indptr[:-1], axis=0)
    hop2 = (A @ mean) / deg
    dst = np.repeat(np.arange(C["n"]), np.diff(A.indptr))
    dist = np.bincount(dst, np.linalg.norm(C["xy"][A.indices] - C["xy"][dst], axis=1), C["n"]) / deg.ravel()
    n_r30 = np.bincount(C["G"]["r30"]["dst"], minlength=C["n"])
    n_con = np.diff(C["CON"].indptr)
    return np.c_[X, mean, mx, std, hop2, np.log1p(n_r30), np.log1p(n_con), dist / 30.0].astype(np.float32)

TOPO_NAMES = [f"{g}_{f}" for g in ("contact", "r30") for f in
              ("log_degree", "log_triangles", "log_open_centre", "log_open_end", "clustering", "log_two_hop")] + ["contact_harmonic"]

def motif_block(A):
    """Undirected 3-node motif roles, clustering and 2-hop size for one image graph (0/1 symmetric CSR)."""
    A = A.tocsr().astype(np.float64)
    n = A.shape[0]
    deg = np.asarray(A.sum(1)).ravel()
    A2 = (A @ A).tocsr()
    tri = np.asarray(A2.multiply(A).sum(1)).ravel() / 2
    wedges = deg * (deg - 1) / 2
    centre = wedges - tri                                       # open triads, cell in the middle
    end = A @ np.maximum(deg - 1, 0) - 2 * tri                  # open triads, cell at an end
    clus = np.where(wedges > 0, tri / np.maximum(wedges, 1), 0.0)
    R = (A2 + A + sp.eye(n, format="csr")).tocsr()
    two_hop = np.diff(R.indptr) - deg - 1
    return np.c_[np.log1p(deg), np.log1p(tri), np.log1p(centre), np.log1p(end), clus, np.log1p(two_hop)]

def harmonic(A):
    D = shortest_path(A, method="D", unweighted=True, directed=False)
    with np.errstate(divide="ignore"):
        inv = np.where(np.isfinite(D) & (D > 0), 1.0 / D, 0.0)
    return inv.sum(1) / max(A.shape[0] - 1, 1)

def topology_features(C, A30):
    out = np.zeros((C["n"], len(TOPO_NAMES)), np.float32)
    for a, b in zip(C["starts"][:-1], C["starts"][1:]):
        Ac = C["CON"][a:b, a:b]
        out[a:b] = np.c_[motif_block(Ac), motif_block(A30[a:b, a:b]), harmonic(Ac)]
    return out

GSTATS = {}
for key, C in CO.items():
    C["G"] = {}
    for gn, (kind, par) in GRAPHS.items():
        A = adjacency(C, kind, par)
        C["G"][gn], GSTATS[(key, gn)] = edge_store(C, A, gn == "r30")
        if gn == "knn10":
            A10 = A
        if gn == "r30":
            A30 = A
        del A
    C["XS"]["nbr"] = nbr_features(C, A10)
    T = topology_features(C, A30)
    C["TOPO_RAW"] = T
    C["XS"]["topo"] = np.nan_to_num(harm(T))
    C["XS"]["own_topo"] = np.c_[C["XS"]["cohort"], C["XS"]["topo"]]
    C["XS"]["nbr_topo"] = np.c_[C["XS"]["nbr"], C["XS"]["topo"]]
    del A10, A30
    gc.collect()
json.dump({f"{k}|{gn}": v for (k, gn), v in GSTATS.items()}, open(f"{OUT}/graph_stats.json", "w"), indent=1)
rows = []
for key, C in CO.items():
    for i, p in enumerate(C["fine"]):
        m = C["y"]["fine"] == i
        r = dict(cohort=key, fine=p, coarse=C["cmap"][p] or "unlabelled", n=int(m.sum()))
        r.update({f: float(np.median(C["TOPO_RAW"][m, j])) for j, f in enumerate(TOPO_NAMES)})
        rows.append(r)
pd.DataFrame(rows).to_csv(f"{OUT}/topology_by_label.csv", index=False)
print(pd.DataFrame(GSTATS).T.round(2).to_string())
tick("graphs and neighbour features")

def combine(A, B, gnames):
    """Jackson + Danenberg as one bank of images (coarse labels only)."""
    nA, eA = A["n"], {gn: len(A["G"][gn]["src"]) for gn in gnames}
    C = dict(key="Both", n=nA + B["n"], n_img=A["n_img"] + B["n_img"], starts=np.r_[A["starts"][:-1], B["starts"] + nA],
             patient=np.r_[A["patient"], B["patient"]], cohort=np.r_[np.zeros(nA, np.int8), np.ones(B["n"], np.int8)],
             XS={k: np.r_[A["XS"][k], B["XS"][k]] for k in A["XS"]}, y={"coarse": np.r_[A["y"]["coarse"], B["y"]["coarse"]]}, G={})
    for gn in gnames:
        a, b = A["G"][gn], B["G"][gn]
        e = dict(src=np.r_[a["src"], b["src"] + nA].astype(np.int32), dst=np.r_[a["dst"], b["dst"] + nA].astype(np.int32),
                 estarts=np.r_[a["estarts"][:-1], b["estarts"] + eA[gn]])
        if "ea" in a:
            e["ea"] = np.r_[a["ea"], b["ea"]]
        C["G"][gn] = e
    return C

## Models

In [ ]:
def seg_sum(v, idx, n):
    return torch.zeros((n,) + tuple(v.shape[1:]), device=v.device, dtype=v.dtype).index_add_(0, idx, v)

class GCNConv(nn.Module):
    """Kipf: symmetric normalisation with self-loops."""
    def __init__(s, din, dout):
        super().__init__()
        s.lin = nn.Linear(din, dout)
    def forward(s, h, src, dst, ea):
        n = h.shape[0]
        deg = 1 + torch.bincount(dst, minlength=n).to(h.dtype)
        z = s.lin(h)
        return z / deg[:, None] + seg_sum(z[src] * (deg[src] * deg[dst]).rsqrt()[:, None], dst, n)

class SAGEConv(nn.Module):
    """Self weight + neighbour aggregate. plus=True: mean, max and SD + log degree."""
    def __init__(s, din, dout, plus=False):
        super().__init__()
        s.plus = plus
        s.root = nn.Linear(din, dout)
        s.nb = nn.Linear(din * (3 if plus else 1) + (1 if plus else 0), dout, bias=False)
    def forward(s, h, src, dst, ea):
        n = h.shape[0]
        m = h[src]
        deg = torch.bincount(dst, minlength=n).to(h.dtype)[:, None]
        mean = seg_sum(m, dst, n) / deg.clamp(min=1)
        parts = [mean]
        if s.plus:
            std = (seg_sum(m * m, dst, n) / deg.clamp(min=1) - mean * mean).clamp(min=0).add(1e-6).sqrt()
            mx = torch.full_like(mean, float("-inf")).scatter_reduce(0, dst[:, None].expand_as(m), m, reduce="amax",
                                                                     include_self=True)
            mx = torch.where(deg > 0, mx, torch.zeros_like(mx))
            parts += [mx, std, torch.log1p(deg)]
        return s.root(h) + s.nb(torch.cat(parts, 1))

class GATv2Conv(nn.Module):
    """GATv2 attention with edge features, plus a self weight."""
    def __init__(s, din, dout, heads=4, edim=2):
        super().__init__()
        s.H, s.Cd = heads, dout // heads
        s.ws, s.wd, s.we = nn.Linear(din, dout), nn.Linear(din, dout, bias=False), nn.Linear(edim, dout, bias=False)
        s.att = nn.Parameter(torch.empty(heads, s.Cd))
        nn.init.xavier_uniform_(s.att)
        s.root = nn.Linear(din, dout)
    def forward(s, h, src, dst, ea):
        n, H, Cd = h.shape[0], s.H, s.Cd
        xs = s.ws(h)
        e = (F.leaky_relu(xs[src] + s.wd(h)[dst] + s.we(ea), 0.2).view(-1, H, Cd) * s.att).sum(-1)    # E x H
        emax = torch.full((n, H), float("-inf"), device=h.device).scatter_reduce(
            0, dst[:, None].expand(-1, H), e.detach(), reduce="amax", include_self=True)
        a = torch.exp(e - emax[dst])
        a = a / seg_sum(a, dst, n)[dst].clamp(min=1e-16)
        return seg_sum(xs[src].view(-1, H, Cd) * a[..., None], dst, n).reshape(n, H * Cd) + s.root(h)

class Net(nn.Module):
    def __init__(s, spec, din, ncls, h=HID, p=0.1):
        super().__init__()
        s.kind, s.jk, L = spec["kind"], spec["jk"], spec["layers"]
        dims = [din] + [h] * L
        mk = {"mlp": nn.Linear, "gcn": GCNConv, "sage": SAGEConv, "sage+": lambda a, b: SAGEConv(a, b, plus=True),
              "gat": GATv2Conv}[s.kind]
        s.convs = nn.ModuleList([mk(a, b) for a, b in zip(dims[:-1], dims[1:])])
        s.dp = nn.Dropout(p)
        s.out = nn.Linear(h * (L if s.jk else 1), ncls)
    def forward(s, x, src, dst, ea):
        hs, h = [], x
        for conv in s.convs:
            h = s.dp(F.relu(conv(h) if s.kind == "mlp" else conv(h, src, dst, ea)))
            hs.append(h)
        return s.out(torch.cat(hs, 1) if s.jk else h)

TABLE_ARMS = {"Table": "cohort", "Table + nbr": "nbr", "Topology": "topo", "Table + topo": "own_topo",
              "Table + nbr + topo": "nbr_topo"}
NET_ARMS = {}
IMGZ_ONLY_COARSE = set()

def class_weights(y, ncls):
    y = y[y >= 0]
    cnt = np.bincount(y, minlength=ncls).astype(np.float64)
    w = np.where(cnt > 0, 1 / np.sqrt(np.maximum(cnt, 1)), 0.0)
    return w / w[y].mean()                                          # mean weight 1 over training cells

def cells_of(C, imgs):
    st = C["starts"]
    return np.concatenate([np.arange(st[i], st[i + 1]) for i in imgs])

def batch_arrays(C, imgs, gname, xkey, task):
    st = C["starts"]
    node = cells_of(C, imgs)
    b = dict(node=node, x=C["XS"][xkey][node], y=C["y"][task][node], src=None, dst=None, ea=None)
    if gname:
        E = C["G"][gname]
        es = E["estarts"]
        eid = np.concatenate([np.arange(es[i], es[i + 1]) for i in imgs])
        off = np.cumsum(np.r_[0, (st[imgs + 1] - st[imgs])[:-1]]) - st[imgs]
        shift = np.repeat(off, es[imgs + 1] - es[imgs])
        b["src"], b["dst"] = E["src"][eid] + shift, E["dst"][eid] + shift
        if "ea" in E:
            b["ea"] = E["ea"][eid]
    return b

def to_dev(b):
    return {k: (torch.from_numpy(v).to(dev) if isinstance(v, np.ndarray) and k != "node" else v) for k, v in b.items()}

def fwd(net, b):
    return net(b["x"], b["src"], b["dst"], b["ea"])

@torch.no_grad()
def val_loss(net, spec, C, task, imgs, w):
    net.eval()
    num = den = 0.0
    for s in range(0, len(imgs), EVAL_IMAGES):
        b = to_dev(batch_arrays(C, imgs[s:s + EVAL_IMAGES], spec["graph"], spec["x"], task))
        ok = b["y"] >= 0
        if ok.any():
            yy = b["y"][ok]
            num += F.cross_entropy(fwd(net, b)[ok], yy, weight=w, reduction="sum").item()
            den += w[yy].sum().item()
    return num / max(den, 1e-12)

def fit_net(spec, C, task, ncls, tr, va, seed):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    w = torch.tensor(class_weights(C["y"][task][cells_of(C, tr)], ncls), dtype=torch.float32, device=dev)
    net = Net(spec, C["XS"][spec["x"]].shape[1], ncls).to(dev)
    opt = torch.optim.AdamW(net.parameters(), lr=2e-3, weight_decay=1e-4)
    best, state, bad, ep = np.inf, None, 0, 0
    for ep in range(MAX_EPOCHS):
        net.train()
        perm = rng.permutation(tr)
        for s in range(0, len(perm), BATCH_IMAGES):
            b = to_dev(batch_arrays(C, perm[s:s + BATCH_IMAGES], spec["graph"], spec["x"], task))
            if not (b["y"] >= 0).any():
                continue
            loss = F.cross_entropy(fwd(net, b), b["y"], weight=w, ignore_index=-1)
            opt.zero_grad()
            loss.backward()
            opt.step()
        v = val_loss(net, spec, C, task, va, w)
        if v < best - 1e-4:
            best, bad = v, 0
            state = {k: t.detach().clone() for k, t in net.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    net.load_state_dict(state)
    net.eval()
    return net, ep + 1, best

@torch.no_grad()
def predict_net(net, spec, C, imgs, task):
    out = np.full(C["n"], -1, np.int64)
    for s in range(0, len(imgs), EVAL_IMAGES):
        ba = batch_arrays(C, imgs[s:s + EVAL_IMAGES], spec["graph"], spec["x"], task)
        out[ba["node"]] = fwd(net, to_dev(ba)).argmax(1).cpu().numpy()
    return out

def fit_table(xkey, C, task, ncls, tr, seed):
    idx = cells_of(C, tr)
    idx = idx[C["y"][task][idx] >= 0]
    rs = np.random.default_rng(seed)
    pools = [idx[C["cohort"][idx] == c] for c in (0, 1)] if C["key"] == "Both" else [idx]
    fit = np.sort(np.concatenate([rs.choice(p, min(N_FIT, len(p)), replace=False) for p in pools if len(p)]))
    y = C["y"][task][fit]
    mdl = lgb.LGBMClassifier(random_state=seed, **LGB).fit(C["XS"][xkey][fit], y, sample_weight=class_weights(y, ncls)[y])
    return mdl, len(fit)

def predict_table(mdl, xkey, C, imgs, task):
    out = np.full(C["n"], -1, np.int64)
    node = cells_of(C, imgs)
    node = node[C["y"][task][node] >= 0]
    out[node] = mdl.classes_[mdl.predict_proba(C["XS"][xkey][node]).argmax(1)]
    return out

## Scoring (patient bootstrap over the test patients)

In [ ]:
def scorer(C, imgs, task, ncls, seed):
    node = cells_of(C, imgs)
    node = node[C["y"][task][node] >= 0]
    pats, pc = np.unique(C["patient"][node], return_inverse=True)
    W = np.random.default_rng(seed).multinomial(len(pats), np.full(len(pats), 1 / len(pats)), size=N_BOOT).astype(np.float64)
    y = C["y"][task][node]
    return dict(node=node, y=y, pc=pc, npat=len(pats), W=W, ncls=ncls, present=np.bincount(y, minlength=ncls) > 0)

def metrics(M, ncls, present):
    M = M.reshape(-1, ncls, ncls)
    tp = np.einsum("bii->bi", M)
    sup, prd = M.sum(2), M.sum(1)
    with np.errstate(invalid="ignore", divide="ignore"):
        f1 = np.where(sup + prd > 0, 2 * tp / (sup + prd), 0.0)
        rec = np.where(sup > 0, tp / sup, np.nan)
        out = dict(macro_f1=f1[:, present].mean(1), bal_acc=np.nanmean(rec[:, present], 1), acc=tp.sum(1) / M.sum((1, 2)))
    return out, f1

def score(S, pred_full):
    p = pred_full[S["node"]]
    assert (p >= 0).all(), "missing predictions"
    k = S["ncls"]
    cm = np.bincount((S["pc"] * k + S["y"]) * k + p, minlength=S["npat"] * k * k).reshape(S["npat"], k * k).astype(np.float64)
    tot = cm.sum(0, keepdims=True)
    pt, f1pt = metrics(tot, k, S["present"])
    bt, f1bt = metrics(S["W"] @ cm, k, S["present"])
    return {m: float(v[0]) for m, v in pt.items()}, bt, f1pt[0], f1bt, tot.reshape(k, k)

def ci(v):
    v = np.asarray(v)[np.isfinite(v)]
    return float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))

def labels_of(task, test):
    return COARSE if task == "coarse" else CO[test]["fine"]

## Replicates × tasks × training sets (+ the SAGE+ sweep after replicate 0)

In [ ]:
UNITS = [("fine", "Danenberg"), ("fine", "Jackson"), ("coarse", "Jackson"), ("coarse", "Danenberg"), ("coarse", "Both")]
EST, SKIPPED, SWEEP_ROWS = {}, [], []
D_TRVA_PAT = np.unique(CD["patient"][CD["split0"] != "test"])

def split_images(C, cell_split):
    s = cell_split[C["starts"][:-1]]
    return {k: np.flatnonzero(s == k) for k in ("train", "val", "test")}

def make_splits(REP):
    jp, jcode = np.unique(CJ["patient"], return_inverse=True)
    u = np.random.default_rng(SEED + 11 + REP).random(len(jp))
    J_SPLIT = np.where(u < 0.7, "train", np.where(u < 0.8, "val", "test"))[jcode]
    uv = np.random.default_rng(SEED + 21 + REP).random(len(D_TRVA_PAT))
    dval = set(D_TRVA_PAT[uv < 0.125])
    D_SPLIT = np.where(CD["split0"] == "test", "test", np.where(pd.Series(CD["patient"]).isin(dval).values, "val", "train"))
    IMG = {"Jackson": split_images(CJ, J_SPLIT), "Danenberg": split_images(CD, D_SPLIT)}
    IMG["Both"] = {s: np.r_[IMG["Jackson"][s], IMG["Danenberg"][s] + CJ["n_img"]] for s in ("train", "val")}
    for c in ("Jackson", "Danenberg"):
        print(c, {s: (len(v), len(set(CO[c]["patient"][cells_of(CO[c], v)]))) for s, v in IMG[c].items()}, "(images, patients)")
    return IMG

def run_arm(name, task, src, C, ncls, tests, IMG, SCR, seed, spec=None):
    """Fit one arm on C (training set src) and score it on each test cohort. Returns score dicts."""
    t0 = time.time()
    if name in TABLE_ARMS and spec is None:
        xkey = TABLE_ARMS[name]
        mdl, n_train = fit_table(xkey, C, task, ncls, IMG[src]["train"], seed)
        preds = {t: predict_table(mdl, xkey, CO[t], IMG[t]["test"], task) for t in tests}
        n_ep, best = np.nan, np.nan
    else:
        spec = spec or NET_ARMS[name]
        net, n_ep, best = fit_net(spec, C, task, ncls, IMG[src]["train"], IMG[src]["val"], seed)
        preds = {t: predict_net(net, spec, CO[t], IMG[t]["test"], task) for t in tests}
        n_train = int((C["y"][task][cells_of(C, IMG[src]["train"])] >= 0).sum())
        del net
        if dev == "cuda":
            torch.cuda.empty_cache()
    res = {t: score(SCR[(task, t)], preds[t]) for t in tests}
    log = dict(task=task, train=src, model=name, epochs=n_ep, best_val_loss=best, n_train=n_train, seconds=time.time() - t0)
    return res, log

def sweep(IMG, SCR):
    for src in ("Jackson", "Danenberg"):
        el = (time.time() - T0) / 60
        proj = EST.get("sweep", 60.0)
        if el + proj > BUDGET_MIN:
            print(f"!!! skipping sweep on {src}: {el:.0f} + {proj:.0f} min > {BUDGET_MIN:.0f}")
            SKIPPED.append(("sweep", src))
            continue
        t_u = time.time()
        for gname in GRAPHS:
            for L in (1, 2, 3):
                spec = dict(NET_ARMS["SAGE+"], graph=gname, layers=L)
                res, log = run_arm(f"SAGE+ {gname} L{L}", "coarse", src, CO[src], len(COARSE), ["Jackson", "Danenberg"],
                                   IMG, SCR, SEED, spec=spec)
                for t, (pt, bt, *_rest) in res.items():
                    for m, v in pt.items():
                        lo, hi = ci(bt[m])
                        SWEEP_ROWS.append(dict(train=src, test=t, graph=gname, layers=L, metric=m, value=v, lo=lo, hi=hi,
                                               epochs=log["epochs"], seconds=log["seconds"]))
            tick(f"sweep {src} {gname}")
        EST["sweep"] = (time.time() - t_u) / 60
        pd.DataFrame(SWEEP_ROWS).to_csv(f"{OUT}/sweep_scores.csv", index=False)
    sw = pd.DataFrame(SWEEP_ROWS)
    if len(sw):
        print(sw[sw.metric == "macro_f1"].pivot_table(index=["train", "test", "graph"], columns="layers", values="value").round(3).to_string())

for REP in REPS:
    tick(f"replicate {REP}")
    IMG = make_splits(REP)
    SCR = {}
    for task in ("fine", "coarse"):
        for t in ("Jackson", "Danenberg"):
            SCR[(task, t)] = scorer(CO[t], IMG[t]["test"], task, len(labels_of(task, t)), SEED + 31 + REP + 1000 * (t == "Danenberg"))
    ROWS, PCL, CONF, LOG, BOOT = [], [], [], [], {}
    for task, src in UNITS:
        el = (time.time() - T0) / 60
        proj = EST.get((task, src), 45.0)
        if el + proj > BUDGET_MIN:
            print(f"!!! skipping rep {REP} {task} / train {src}: {el:.0f} + {proj:.0f} min > {BUDGET_MIN:.0f}")
            SKIPPED.append((REP, task, src))
            continue
        t_u = time.time()
        C = combine(CJ, CD, ["knn10", "r30"]) if src == "Both" else CO[src]
        ncls = len(COARSE) if task == "coarse" else len(CO[src]["fine"])
        tests = ["Jackson", "Danenberg"] if task == "coarse" else [src]
        arms = [a for a in list(TABLE_ARMS) + list(NET_ARMS) if task == "coarse" or a not in IMGZ_ONLY_COARSE]
        for name in arms:
            res, log = run_arm(name, task, src, C, ncls, tests, IMG, SCR, SEED + REP)
            LOG.append(dict(rep=REP, **log))
            for t, (pt, bt, f1pt, f1bt, cm) in res.items():
                BOOT[(task, src, t, name)] = bt
                S = SCR[(task, t)]
                for m, v in pt.items():
                    lo, hi = ci(bt[m])
                    ROWS.append(dict(rep=REP, task=task, train=src, test=t, model=name, metric=m, value=v, lo=lo, hi=hi,
                                     n_cells=len(S["node"]), n_patients=S["npat"]))
                labs = labels_of(task, t)
                for c in np.flatnonzero(S["present"]):
                    lo, hi = ci(f1bt[:, c])
                    PCL.append(dict(rep=REP, task=task, train=src, test=t, model=name, label=labs[c], support=int(cm[c].sum()),
                                    f1=float(f1pt[c]), lo=lo, hi=hi))
                if task == "coarse":
                    for i in range(ncls):
                        for j in range(ncls):
                            if cm[i, j]:
                                CONF.append(dict(rep=REP, train=src, test=t, model=name, true=labs[i], pred=labs[j], n=int(cm[i, j])))
            tick(f"rep {REP} {task} / train {src}: {name} "
                 + ", ".join(f"{t} F1 {r[0]['macro_f1']:.3f}" for t, r in res.items()))
        if src == "Both":
            del C
            gc.collect()
        EST[(task, src)] = (time.time() - t_u) / 60
        pd.DataFrame(LOG).to_csv(f"{OUT}/training_log_rep{REP}.csv", index=False)
    RES = pd.DataFrame(ROWS)
    RES.to_csv(f"{OUT}/scores_rep{REP}.csv", index=False)
    pd.DataFrame(PCL).to_csv(f"{OUT}/perclass_rep{REP}.csv", index=False)
    pd.DataFrame(CONF).to_csv(f"{OUT}/confusion_rep{REP}.csv", index=False)
    # paired differences (same bootstrap draws within a task x training set x test set)
    DIFF = []
    for (task, src, t, name), bt in BOOT.items():
        for base in ("Table", "Table + nbr", "Topology"):
            kb = (task, src, t, base)
            if base == name or kb not in BOOT:
                continue
            for m in ("macro_f1", "bal_acc"):
                d = RES[(RES.task == task) & (RES.train == src) & (RES.test == t) & (RES.metric == m)].set_index("model").value
                lo, hi = ci(bt[m] - BOOT[kb][m])
                DIFF.append(dict(rep=REP, task=task, train=src, test=t, model=name, base=base, metric=m,
                                 diff=float(d[name] - d[base]), lo=lo, hi=hi,
                                 call="better" if lo > 0 else "worse" if hi < 0 else "n.s."))
    pd.DataFrame(DIFF).to_csv(f"{OUT}/diffs_rep{REP}.csv", index=False)
    if len(RES):
        print(f"--- replicate {REP}: macro-F1")
        print(RES[RES.metric == "macro_f1"].pivot_table(index=["task", "train", "test"], columns="model", values="value")
              .round(3).to_string())
    tick(f"replicate {REP} done")
    if REP == REPS[0] and SWEEP:
        sweep(IMG, SCR)

print("skipped units:", SKIPPED if SKIPPED else "none")
json.dump({"skipped": [list(map(str, s)) for s in SKIPPED], "reps": REPS, "budget_min": BUDGET_MIN,
           "minutes": (time.time() - T0) / 60, "device": dev}, open(f"{OUT}/run_info.json", "w"))

## Across replicates: macro-F1 and balanced accuracy (mean ± SD)

In [ ]:
fs = sorted(glob.glob(f"{OUT}/scores_rep*.csv"))
if fs:
    ALL = pd.concat([pd.read_csv(f) for f in fs])
    AGG = ALL.groupby(["task", "train", "test", "model", "metric"]).value.agg(["mean", "std", "count"]).reset_index()
    AGG.to_csv(f"{OUT}/summary_scores.csv", index=False)
    print(AGG[AGG.metric == "macro_f1"].pivot_table(index=["task", "train", "test"], columns="model", values="mean")
          .round(3).to_string())